### Using ChromaDB for the vector database and wrapping the final response with an LLM is a classic, highly effective RAG (Retrieval-Augmented Generation) setup for customer support.

# 1. Install dependencies

### !pip install chromadb pandas anthropic sentence-transformers python-dotenv -q

# 2. Load and inspect the dataset

In [30]:
import pandas as pd

df = pd.read_csv("customer_support_tickets.csv")
df = df.dropna(subset=["Ticket Description"])  # need text to embed
print(df.shape)
df.head()

(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


# 3. Build a "document" per ticket for embedding
### Combine the fields that matter for semantic search into one text blob per ticket, and keep the rest as metadata (for filtering/display).

In [31]:
def build_document(row):
    return (
        f"Ticket Subject: {row['Ticket Subject']}\n"
        f"Ticket Type: {row['Ticket Type']}\n"
        f"Description: {row['Ticket Description']}\n"
        f"Resolution: {row.get('Resolution', 'N/A')}"
    )

df["doc_text"] = df.apply(build_document, axis=1)

def build_metadata(row):
    return {
        "ticket_id": str(row["Ticket ID"]),
        "product": str(row.get("Product Purchased", "")),
        "ticket_type": str(row.get("Ticket Type", "")),
        "priority": str(row.get("Ticket Priority", "")),
        "status": str(row.get("Ticket Status", "")),
        "channel": str(row.get("Ticket Channel", "")),
    }

metadatas = df.apply(build_metadata, axis=1).tolist()
documents = df["doc_text"].tolist()
ids = [str(x) for x in df["Ticket ID"].tolist()]

# 4. Set up ChromaDB with an embedding function
### Using a local sentence-transformers model keeps this free and fast (no API calls needed just for embeddings).

In [32]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="./chroma_store")

collection = client.get_or_create_collection(
    name="support_tickets",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# 5. Ingest data in batches (Chroma has batch size limits)

In [33]:
from tqdm import tqdm

BATCH = 500

for i in tqdm(range(0, len(documents), BATCH), desc="Adding documents"):
    collection.add(
        documents=documents[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
        ids=ids[i:i+BATCH]
    )

print("Total docs in collection:", collection.count())

Adding documents:   0%|          | 0/17 [00:00<?, ?it/s]

Adding documents: 100%|██████████| 17/17 [00:49<00:00,  2.90s/it]

Total docs in collection: 8469


# 6. Retrieval function

In [34]:
def retrieve(query, n_results=5, where=None):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where  # e.g. {"priority": "high"}
    )
    hits = []
    for doc, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        hits.append({"text": doc, "metadata": meta, "distance": dist})
    return hits

# 7. LLM wrapper to generate the final answer
### Using the Cerebras API — pass retrieved context + user query, get a grounded answer.

In [41]:
import os
import json
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Secure client initialization with standard OpenAI schema
client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

# CHANGED: Updated default model to a valid Cerebras ID ("llama-3.3-70b" or "gpt-oss-120b")
def call_llm_with_retry(messages, model="gpt-oss-120b", temperature=0.2, max_retries=5):
    """
    Executes a chat completion call with explicit exponential backoff 
    to handle 429 (Too Many Requests) errors gracefully.
    """
    delay = 1
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature
            )
            # FIX: Correctly unpack the message text from standard OpenAI response schema
            return response.choices[0].message.content
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                time.sleep(delay)
                delay *= 2  # Exponentially back off the hit rate
                continue
            raise e

def extract_metadata_and_hyde(query):
    """
    Advanced RAG: Evaluates the raw query using an LLM to generate a 
    hypothetical document profile (HyDE) and pull filter attributes.
    """
    system_prompt = (
        "You are an AI router for a customer support database. Parse the user query and respond ONLY with a JSON object containing:\n"
        "1. 'hypothetical_ticket': A realistic version of what a historic support ticket description looks like for this problem.\n"
        "2. 'where_filter': A dictionary containing standard filters for ChromaDB metadata (e.g., {'product': 'Dell XPS'}) if named, otherwise {}.\n"
        "Valid products: 'GoPro Hero', 'LG Smart TV', 'Dell XPS', 'Microsoft Office', 'Autodesk AutoCAD'.\n"
        "Do not include markdown wrappers, return raw valid JSON."
    )
    
    try:
        raw_json = call_llm_with_retry([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"User Query: {query}"}
        ], temperature=0.1)
        
        cleaned_json = raw_json.strip().replace("```json", "").replace("```", "")
        parsed = json.loads(cleaned_json)
        return parsed.get("hypothetical_ticket", query), parsed.get("where_filter", {})
    except Exception:
        # Fall back gracefully to standard raw user text on serialization errors
        return query, {}

def generate_answer(query, hits):
    if not hits:
        return "I couldn't find any historical support documentation matching that request."

    context = "\n\n---\n\n".join(
        f"[Ticket {h['metadata']['ticket_id']} | Product: {h['metadata']['product']}] {h['text']}" for h in hits
    )

    system_prompt = (
        "You are an expert customer support agent. Use the provided context from historical tickets "
        "to assist the user with their current problem. Extract the direct technical strategies that worked.\n"
        "Rules:\n"
        "- Base your answer strict on the historical context.\n"
        "- If historical resolution entries look like filler text, interpret the technical context instead.\n"
        "- Lower the noise, maintain structured clarity, and do not guess if it lacks relevance."
    )

    user_prompt = f"Historical Tickets Context:\n{context}\n\nUser Current Problem: {query}"

    return call_llm_with_retry([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ], temperature=0.2) # Lower temperature drastically reduces hallucination rates

# 8. RAG pipeline function tying it together

In [42]:
def rag_query(query, n_results=3, verbose=False):
    # 1. Expand standard input using the Advanced Router
    hyde_query, search_filters = extract_metadata_and_hyde(query)
    where_clause = search_filters if search_filters else None
    
    # 2. Extract context dynamically
    results = collection.query(
        query_texts=[hyde_query],
        n_results=n_results,
        where=where_clause
    )
    
    hits = []
    if results and "documents" in results and len(results["documents"]) > 0:
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
            hits.append({"text": doc, "metadata": meta, "distance": dist})
            
    if verbose:
        print(f"\n[DEBUG METADATA] Query Filters Used: {where_clause}")
        print(f"[DEBUG HyDE] Generated Pseudo-Doc Target: {hyde_query}")
        for h in hits:
            print(f" -> Hit: Ticket ID {h['metadata']['ticket_id']} (Cosine Distance: {h['distance']:.3f})")
            
    # 3. Grounded response delivery
    answer = generate_answer(query, hits)
    return answer

# 9. CLI chatbot loop

In [43]:
def run_cli():
    print("=" * 65)
    print("🚀 Advanced Customer Support RAG Engine Initiated")
    print("Commands: Type 'exit' to quit | Type 'debug' to toggle background metadata view.")
    print("=" * 65)
    
    debug_mode = False
    while True:
        try:
            query = input("\n👤 You: ").strip()
            if query.lower() in ("exit", "quit"):
                print("Closing support console connection. Goodbye!")
                break
            if not query:
                continue
            if query.lower() == "debug":
                debug_mode = not debug_mode
                print(f"⚙️ Debug trace mode altered to: {debug_mode}")
                continue
            
            print("🤖 Processing vector match optimization...", end="\r")
            answer = rag_query(query, n_results=3, verbose=debug_mode)
            print(f"🤖 Bot: {answer}")
            
        except KeyboardInterrupt:
            print("\nForce close detected. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error handling pipeline execution path: {e}")

run_cli()

🚀 Advanced Customer Support RAG Engine Initiated
Commands: Type 'exit' to quit | Type 'debug' to toggle background metadata view.
🤖 Bot: **What the historical tickets tell us**

- All three tickets describe a *screen‑related hardware problem* (flickering or black screen) on different products.  
- The “Resolution” fields are empty (`nan`) and the written answers (“Turn‑on the lights”, “It’s possible to run this with no problem”) do not contain actionable troubleshooting steps.  
- Therefore the historical data does **not** provide a concrete fix that we can copy‑paste.

**Standard, proven steps for a black‑screen / flickering issue**

| Step | What to do | Why it helps |
|------|------------|--------------|
| 1️⃣ Power cycle the device | • Unplug the device (or remove the battery if removable).<br>• Wait ≈ 30 seconds.<br>• Plug it back in and power on. | Clears temporary glitches and forces a fresh boot. |
| 2️⃣ Verify the power source | • Ensure the outlet works (test with another dev